In [1]:
import pandas as pd
from sklearn import preprocessing
import numpy as np
import random

In [14]:
#aids dataset preprocessing, from https://github.com/tobhatt/CorNet/blob/main/real_world_exp/load_rct_data.py

def sample_rct(n_unc):
    df = pd.read_csv("aids.csv", header=0, index_col=0)
    
    cd4_baseline = df['cd40']
    cd4_20 = df['cd420']
    
    outcome = (cd4_20 - cd4_baseline).values
    outcome_norm = preprocessing.scale(outcome)
    
    cov_cont = df[['age', 'wtkg', 'cd40', 'karnof', 'cd80']]
    cov_cont_norm = preprocessing.scale(cov_cont)
    
    cov_bin = df[['gender', 'homo', 'race', 'drugs', 'symptom', 'str2', 'hemo']]
    cov_bin_val = cov_bin.values 
    t = df[['arms']].values
    
    data = np.concatenate((cov_cont_norm, cov_bin_val, t.reshape(-1,1), outcome_norm.reshape(-1,1)), axis=1)
    data.shape
    
    #Only focus on one arm (0=zidovudine, 1=zidovudine and didanosine, 2=zidovudine and zalcitabine,3=didanosine)
    t_1 = 2
    t_0 = 0
    t_ind = (t == t_0) + (t == t_1)
    
    data_rct = data[t_ind.flatten()]
    #change treatment sign to 1
    data_rct[:,-2] = np.where(data_rct[:,-2] == 2, 1, 0)
    
    return pd.DataFrame(data_rct, columns=['age', 'wtkg', 'cd40', 'karnof', 'cd80', 'gender', 'homo', 'race', 'drugs', 'symptom', 'str2', 'hemo', 't', 'y'])
    #All data
    x_all = data_rct[:,:-2]
    t_all = data_rct[:,-2].reshape(-1,1)
    y_all = data_rct[:,-1].reshape(-1,1)

    #UNC selection
    ind_unc = random.sample(range(x_all.shape[0]), n_unc)
    x_unc = x_all[ind_unc, ]
    t_unc = t_all[ind_unc, ].reshape(-1,1)
    y_unc = y_all[ind_unc, ].reshape(-1,1)
    
    x_not_unc = np.delete(x_all, ind_unc, axis = 0)
    t_not_unc = np.delete(t_all, ind_unc)
    y_not_unc = np.delete(y_all, ind_unc)
    
    #CONF selection - balanced gender - take all females and sample male s.t. ~ balanced
    #Among males, introduce confounding
    ind_f = (x_not_unc[:, 5] == 0)
    ind_m = (x_not_unc[:, 5] == 1)
    
    ind_m_t = (t_not_unc == 1) * ind_m
    mean = y_not_unc[ind_m_t].mean()
    std = y_not_unc[ind_m_t].std()
    ind_m_t_upper = y_not_unc[ind_m_t] > mean
    
    x_m_t_upper = x_not_unc[ind_m_t,:][ind_m_t_upper,:]
    t_m_t_upper = t_not_unc[ind_m_t][ind_m_t_upper]
    y_m_t_upper = y_not_unc[ind_m_t][ind_m_t_upper]

    ind_m_c = (t_not_unc == 0) * ind_m
    mean = y_not_unc[ind_m_c].mean()
    std = y_not_unc[ind_m_c].std()
    ind_m_c_lower = y_not_unc[ind_m_c] < mean

    x_m_c_lower = x_not_unc[ind_m_c,:][ind_m_c_lower,:]
    t_m_c_lower = t_not_unc[ind_m_c][ind_m_c_lower]
    y_m_c_lower = y_not_unc[ind_m_c][ind_m_c_lower]

    x_f = x_not_unc[ind_f,:]
    t_f = t_not_unc[ind_f]
    y_f = y_not_unc[ind_f]
    
    x_conf = np.concatenate((x_m_t_upper, x_m_c_lower, x_f))
    t_conf = np.concatenate((t_m_t_upper, t_m_c_lower, t_f)).reshape(-1,1)
    y_conf = np.concatenate((y_m_t_upper, y_m_c_lower, y_f)).reshape(-1,1)

    
    x_test = x_not_unc
    t_test = t_not_unc.reshape(-1,1)
    y_test = y_not_unc.reshape(-1,1)

    return {'x_unc': x_unc, 't_unc': t_unc, 'y_unc': y_unc, 'x_conf': x_conf, 't_conf': t_conf, 'y_conf': y_conf, 'x_test': x_test, 't_test': t_test, 'y_test': y_test, 'x_all': x_all, 't_all': t_all, 'y_all': y_all}



In [15]:
df = pd.read_csv("aids.csv", header=0, index_col=0)

In [24]:
df

,pidnum,age,wtkg,hemo,homo,drugs,karnof,oprior,z30,zprior,...,offtrt,cd40,cd420,cd496,r,cd80,cd820,cens,days,arms
1,10056,48,89.8128,0,0,0,100,0,0,1,...,0,422,477,660.0,1,566,324,0,948,2
2,10059,61,49.4424,0,0,0,90,0,1,1,...,0,162,218,NaN,0,392,564,1,1002,3
3,10089,45,88.4520,0,1,1,90,0,1,1,...,1,326,274,122.0,1,2063,1893,0,961,3
4,10093,47,85.2768,0,1,0,100,0,1,1,...,0,287,394,NaN,0,1590,966,0,1166,3
5,10124,43,66.6792,0,1,0,100,0,1,1,...,0,504,353,660.0,1,870,782,0,1090,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2135,990021,21,53.2980,1,0,0,100,0,1,1,...,1,152,109,NaN,0,561,720,0,1091,3
2136,990026,17,102.9672,1,0,0,100,0,1,1,...,1,373,218,NaN,0,1759,1030,0,395,0
2137,990030,53,69.8544,1,1,0,90,0,1,1,...,0,419,364,526.0,1,1391,1041,0,1104,2
2138,990071,14,60.0000,1,0,0,100,0,0,1,...,0,166,169,28.0,1,999,1838,1,465,0


In [20]:
p = sample_rct(100)

In [23]:
p.to_csv('aids_preprocessed.csv', index=False)


In [27]:
p['gender'].astype(int)

0       0
1       1
2       1
3       1
4       1
       ..
1051    1
1052    1
1053    1
1054    1
1055    1
Name: gender, Length: 1056, dtype: int64